In [ ]:
# CREATE CSV

from pathlib import Path

import pandas as pd
from astropy.io import fits
from data_types import Header_Content, S4ACS_Config
from utils import create_csv

config = S4ACS_Config.from_config_file("acs_config.cfg")
nights_folder = config.image_path / ".."

# nights = [
#     night.stem
#     for night in nights_folder.iterdir()
#     if "2026" in night.stem and "test" not in night.stem and "temp" not in night.stem
# ]
nights = [
    "20260731",
    "20260702",
    "20260822",
    "20260719",
    "20260805",
    "20260714",
    "20260823",
    "20260819",
    "20260715",
    "20260803",
]
for night in nights:
    csv_file = Path.cwd() / "_csv" / f"{night}.csv"
    create_csv(csv_file)
    config = S4ACS_Config.from_config_file("acs_config.cfg")
    images_folder = config.image_path / ".." / night
    images_list = [
        name.resolve()
        for name in images_folder.iterdir()
        if name.suffix == ".fits" and "test" not in name.name
    ]

    for image_file in images_list:
        hdr = fits.getheader(image_file)
        hdr_cnt = Header_Content.from_header(hdr)
        df = pd.DataFrame([hdr_cnt.to_csv_format])
        df.to_csv(csv_file, mode="a", index=False, header=False)

In [ ]:
# CREATE REPORT

import pandas as pd
from pathlib import Path

headers = ["Date", "N. images", "Bias", "Flat", "Phot", "Polar"]
report = {k: [] for k in headers}

csv_folder = Path.cwd() / "_ch1_csv"
obs_nights = [night.stem for night in csv_folder.iterdir()]
# obs_nights = [
#     "20260731",
#     "20260702",
#     "20260822",
#     "20260719",
#     "20260805",
#     "20260714",
#     "20260823",
#     "20260819",
#     "20260715",
#     "20260803",
# ]
for night in obs_nights:
    report["Date"].append(night)
    df = pd.read_csv(csv_folder / f"{night}.csv")
    n_images = df.shape[0]
    report["N. images"].append(n_images)
    bias = df[df["OBSTYPE"] == "ZERO"].shape[0]
    report["Bias"].append(bias)
    flat = df[df["OBSTYPE"] == "FLAT"].shape[0]
    report["Flat"].append(flat)
    phot_mode = df[df["INSTMODE"] == "PHOT"].shape[0]
    report["Phot"].append(phot_mode)
    polar_mode = df[df["INSTMODE"] == "POLAR"]
    polar_mode = (
        polar_mode[polar_mode["OBSTYPE"] == "OBJECT"]
        + polar_mode[polar_mode["OBSTYPE"] == "FLAT"]
    )
    report["Polar"].append(polar_mode.shape[0])

report = pd.DataFrame.from_dict(report)
report = report[report["Polar"] > 0]
report = report.sort_values("Date", ascending=False)
report_file = Path.cwd() / "report.csv"
report.to_csv(report_file, index=False)
report = report[["Date", "Polar"]]
# print(report.to_latex(index=False))
print(report["Date"].head(10))


134    20261116
133    20261115
132    20261114
131    20261113
130    20261112
129    20261111
126    20261106
125    20261105
124    20261104
123    20261103
Name: Date, dtype: str
